# OpenPlaque RCA Source-CCTA Seeded Tracer

Clean implementation built from `main` only. Use **Runtime → Run all** once. Google Drive mounts first. The notebook loads source CCTA series **7** and then leaves a backend-free seed picker on screen.

Use the **z / x / y sliders** to move the crosshair to the RCA lumen. Press **Save as OSTIUM** for the proximal seed, then move distally and press **Save as DISTAL**. Finally press **Trace RCA**. No coordinate typing, no Matplotlib widget backend, and no additional cell execution are required.

Research prototype only. Every traced path must be visually checked.

In [ ]:
# ALWAYS FIRST: mount Google Drive.
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive mounted.')

In [ ]:
# Clone the clean branch and install only dependencies needed here.
!rm -rf /content/OpenPlaque
!git clone -q --branch rca-centerline-from-main --single-branch https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
%pip -q install pydicom SimpleITK scipy scikit-image matplotlib ipywidgets

import os, sys, time, shutil
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

SRC = Path('/content/OpenPlaque/src')
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
from openplaque.study import OpenPlaqueStudy
from openplaque.source_centerline import trace_seeded_coronary
print('Dependencies ready. Matplotlib backend:', plt.get_backend())

## Load source CCTA series 7
The study ZIP is copied from Drive to local Colab storage before extraction/scanning.

In [ ]:
ROOT = Path('/content/drive/MyDrive/OpenPlaque')
DRIVE_ZIP = ROOT / 'Full_DICOM.zip'
LOCAL_ZIP = Path('/content/Full_DICOM.zip')
EXTRACT_ROOT = '/content/full_dicom_seeded_rca'
SOURCE_SERIES = 7
if not DRIVE_ZIP.exists():
    raise FileNotFoundError(f'Missing {DRIVE_ZIP}')

t = time.time()
if not LOCAL_ZIP.exists() or LOCAL_ZIP.stat().st_size != DRIVE_ZIP.stat().st_size:
    print(f'Copying Full_DICOM.zip ({DRIVE_ZIP.stat().st_size/1e9:.2f} GB) to local disk...', flush=True)
    shutil.copyfile(DRIVE_ZIP, LOCAL_ZIP)
    print(f'Copy finished in {time.time()-t:.1f}s', flush=True)
else:
    print('Local ZIP already staged.')

shutil.rmtree(EXTRACT_ROOT, ignore_errors=True)
print('Extracting/scanning DICOM locally...', flush=True)
t = time.time()
study = OpenPlaqueStudy(str(LOCAL_ZIP), extract_root=EXTRACT_ROOT)
print(f'Scan finished in {time.time()-t:.1f}s; {len(study.series)} series found.', flush=True)
source_img, source, source_files = study.load_series(SOURCE_SERIES)
print('Source series:', SOURCE_SERIES)
print('Shape zyx:', source.shape)
print('Spacing xyz mm:', source_img.GetSpacing())

## Backend-free RCA seed picker
The crosshair position is controlled by standard Colab sliders. Start near the aortic root, move the crosshair to the center of the proximal RCA lumen, save it as the ostium, then move to a clearly distal RCA location and save the distal seed. The small zoom panel makes fine positioning easier.

In [ ]:
state = {'ostium': None, 'distal': None, 'result': None}
z0 = min(335, source.shape[0]-1)
x0 = source.shape[2]//2
y0 = source.shape[1]//2
z_slider = widgets.IntSlider(value=z0, min=0, max=source.shape[0]-1, step=1, description='z', continuous_update=False, layout=widgets.Layout(width='750px'))
x_slider = widgets.IntSlider(value=x0, min=0, max=source.shape[2]-1, step=1, description='x', continuous_update=False, layout=widgets.Layout(width='750px'))
y_slider = widgets.IntSlider(value=y0, min=0, max=source.shape[1]-1, step=1, description='y', continuous_update=False, layout=widgets.Layout(width='750px'))
ostium_button = widgets.Button(description='Save as OSTIUM', button_style='success')
distal_button = widgets.Button(description='Save as DISTAL', button_style='info')
trace_button = widgets.Button(description='Trace RCA', button_style='warning')
status = widgets.HTML(value='<b>Move z/x/y until the crosshair is centered in the RCA lumen.</b>')
view_out = widgets.Output()
trace_out = widgets.Output()

def current_seed():
    return (int(z_slider.value), int(y_slider.value), int(x_slider.value))

def render_seed_view(*args):
    z,y,x = current_seed()
    with view_out:
        clear_output(wait=True)
        fig,axes = plt.subplots(1,2,figsize=(13,6))
        axes[0].imshow(source[z],cmap='gray',vmin=-200,vmax=800)
        axes[0].axvline(x,lw=1); axes[0].axhline(y,lw=1)
        if state['ostium'] is not None and state['ostium'][0]==z:
            axes[0].scatter([state['ostium'][2]],[state['ostium'][1]],s=100,facecolors='none',linewidths=2,label='ostium')
        if state['distal'] is not None and state['distal'][0]==z:
            axes[0].scatter([state['distal'][2]],[state['distal'][1]],s=100,marker='s',facecolors='none',linewidths=2,label='distal')
        axes[0].set_xlim(0,source.shape[2]-1); axes[0].set_ylim(source.shape[1]-1,0)
        axes[0].set_title(f'Full axial slice z={z}  |  HU at crosshair={float(source[z,y,x]):.0f}')
        axes[0].axis('off')
        r=45
        xlo=max(0,x-r); xhi=min(source.shape[2],x+r+1); ylo=max(0,y-r); yhi=min(source.shape[1],y+r+1)
        axes[1].imshow(source[z,ylo:yhi,xlo:xhi],cmap='gray',vmin=-200,vmax=800,extent=[xlo,xhi-1,yhi-1,ylo])
        axes[1].axvline(x,lw=1); axes[1].axhline(y,lw=1)
        axes[1].set_title('90×90 pixel zoom around crosshair')
        axes[1].set_xlim(xlo,xhi-1); axes[1].set_ylim(yhi-1,ylo)
        plt.tight_layout(); plt.show(); plt.close(fig)

for s in (z_slider,x_slider,y_slider):
    s.observe(render_seed_view,names='value')

def save_seed(which):
    state[which] = current_seed()
    z,y,x = state[which]
    status.value = f'Saved <b>{which}</b> at zyx={state[which]}, HU={float(source[z,y,x]):.0f}. Ostium={state["ostium"]}; distal={state["distal"]}'
    render_seed_view()
ostium_button.on_click(lambda b: save_seed('ostium'))
distal_button.on_click(lambda b: save_seed('distal'))

def physical_to_zyx(pt_xyz):
    x,y,z = source_img.TransformPhysicalPointToContinuousIndex(tuple(float(v) for v in pt_xyz))
    return np.array([z,y,x],dtype=float)

def show_trace(result):
    route=np.asarray(result.points_zyx_voxel)
    margin=np.ceil(12.0/np.asarray(source_img.GetSpacing())[::-1]).astype(int)
    lo=np.maximum(0,np.floor(route.min(axis=0)).astype(int)-margin)
    hi=np.minimum(np.asarray(source.shape),np.ceil(route.max(axis=0)).astype(int)+margin+1)
    crop=source[lo[0]:hi[0],lo[1]:hi[1],lo[2]:hi[2]]
    rr=route-lo
    fig2,axes=plt.subplots(1,3,figsize=(16,5))
    axes[0].imshow(np.max(crop,axis=0),cmap='gray',vmin=-100,vmax=700); axes[0].plot(rr[:,2],rr[:,1],'-',lw=2); axes[0].set_title('Axial projection')
    axes[1].imshow(np.max(crop,axis=1),cmap='gray',vmin=-100,vmax=700,aspect='auto'); axes[1].plot(rr[:,2],rr[:,0],'-',lw=2); axes[1].set_title('Coronal projection')
    axes[2].imshow(np.max(crop,axis=2),cmap='gray',vmin=-100,vmax=700,aspect='auto'); axes[2].plot(rr[:,1],rr[:,0],'-',lw=2); axes[2].set_title('Sagittal projection')
    for d,pt in sorted(result.landmarks_xyz_mm.items()):
        q=physical_to_zyx(pt)-lo
        axes[0].scatter([q[2]],[q[1]],s=70); axes[0].text(q[2]+2,q[1],f'{d:.0f} mm')
        axes[1].scatter([q[2]],[q[0]],s=70); axes[1].text(q[2]+2,q[0],f'{d:.0f} mm')
        axes[2].scatter([q[1]],[q[0]],s=70); axes[2].text(q[1]+2,q[0],f'{d:.0f} mm')
    for a in axes: a.axis('off')
    plt.tight_layout(); plt.show(); plt.close(fig2)

def do_trace(button):
    if state['ostium'] is None or state['distal'] is None:
        status.value='<b>Set both OSTIUM and DISTAL seeds first.</b>'
        return
    with trace_out:
        clear_output(wait=True)
        print('Tracing RCA. Frangi vesselness + shortest path may take 1–3 minutes...',flush=True)
        t=time.time()
        try:
            result=trace_seeded_coronary(source,source_img,state['ostium'],state['distal'])
            state['result']=result
            print(f'Done in {time.time()-t:.1f}s. Centerline length={result.length_mm:.1f} mm')
            print('Landmarks available:',sorted(result.landmarks_xyz_mm))
            if 50.0 not in result.landmarks_xyz_mm:
                print('FAIL: traced path is shorter than 50 mm. Choose a more distal RCA seed.')
            show_trace(result)
            print('PASS only if the line follows the RCA continuously and the 10/50-mm points are anatomically plausible.')
        except Exception as e:
            print('TRACE FAILED:',repr(e))
trace_button.on_click(do_trace)

display(z_slider,x_slider,y_slider,widgets.HBox([ostium_button,distal_button,trace_button]),status,view_out,trace_out)
render_seed_view()
print('Run All is complete. Use the sliders and buttons above; no more cells need to be run.')